# Lazy Arsenal

The notebook uses two ready-made configurations: multi-model `test_playbook_arsenal_router_mode.toml` for Router Mode and `test_playbook_arsenal_model_mode.toml` with one model per server for Model Mode. The constructor only reads TOML and creates the object tree; resources activate on first model access. The commented `arsenal.download()` method can download all resources in advance without starting processes.

In [ ]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession

## Building the object tree

The following cells only read TOML and create objects. They do not download models or llama.cpp and do not start processes.

In [ ]:
router_mode_arsenal = ArsenalSession(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml"
)
# router_mode_arsenal.download()  # Download all resources in advance.

In [ ]:
model_mode_arsenal = ArsenalSession(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)
# model_mode_arsenal.download()  # Download all resources in advance.

# Playbook begin and end

`playbook.begin` enables lazy mode without downloading or starting anything. With `stop_before_begin=True`, it only stops previous processes. Download and startup occur when a specific model is first accessed.

### Model Mode: clean start

Recommended approach: first stop possible old Arsenal processes, access only the required models, and stop the started servers after the playbook.

In [ ]:
zemi.arsenal.begin(model_mode_arsenal, 
    stop_before_begin=True,
    llama_router_mode=False,
)

primary_model = model_mode_arsenal.llamas.primary.models.qwen
secondary_model = model_mode_arsenal.llamas.secondary.models.phi

# At this point both models are downloaded and both servers are ready.

zemi.arsenal.end(model_mode_arsenal, stop_after_end=True)

### Model Mode: start without a preliminary stop

Use this only when the configured ports are known to be free. Ending with `False` intentionally leaves servers running for the next playbook; the final line demonstrates explicit later cleanup.

In [ ]:
zemi.arsenal.begin(model_mode_arsenal, 
    stop_before_begin=False,
    llama_router_mode=False,
)

model_mode_arsenal.llamas.primary.models.qwen

# Servers remain available after the playbook ends.
zemi.arsenal.end(model_mode_arsenal, stop_after_end=False)

# Run later when the servers are no longer needed.
zemi.arsenal.end(model_mode_arsenal, stop_after_end=True)

## Router Mode: clean start

On first access, the parent router starts with an INI preset containing paths to all server models. Subsequent models are downloaded and loaded through `/models/load` without restarting the router.

In [ ]:
zemi.arsenal.begin(router_mode_arsenal, 
    stop_before_begin=True,
    llama_router_mode=True,
)

### Object model of running Arsenal

After `playbook.begin`, lazy activation is enabled but servers are not yet running. The first access through `llamas → models` prepares the selected model. At every level, an object can be accessed by index, string name, or attribute; `config` contains the source TOML table.

In [ ]:
# Llama server: index, name, and attribute access.
primary_by_index = router_mode_arsenal.llamas[0]
primary_by_name = router_mode_arsenal.llamas["primary"]
primary_by_dot = router_mode_arsenal.llamas.primary
assert primary_by_index is primary_by_name is primary_by_dot

# Model: the same three access methods.
qwen_by_index = primary_by_dot.models[0]
qwen_by_name = primary_by_dot.models["qwen"]
qwen_by_dot = primary_by_dot.models.qwen
assert qwen_by_index is qwen_by_name is qwen_by_dot

# Assistant: the same three access methods.
assistant_by_index = qwen_by_dot.assistants[0]
assistant_by_name = qwen_by_dot.assistants["assistant"]
assistant_by_dot = qwen_by_dot.assistants.assistant
assert assistant_by_index is assistant_by_name is assistant_by_dot

# Collections preserve order and support negative indices.
assert router_mode_arsenal.llamas[-1].name == "secondary"
assert primary_by_dot.models[-1].name == "smollm"

{
    "llama_names": list(router_mode_arsenal.llamas.keys()),
    "model_names": list(primary_by_dot.models.keys()),
    "assistant_names": list(qwen_by_dot.assistants.keys()),
    "llama_config": primary_by_dot.config,
    "model_config": qwen_by_dot.config,
    "assistant_config": assistant_by_dot.config,
}

### Ending the playbook

After demonstrating the object model, stop all servers in the configuration.

In [ ]:
zemi.arsenal.end(router_mode_arsenal, stop_after_end=True)

## Router Mode: start on known-free ports

Starting without a preliminary stop is useful when the environment is controlled externally. Servers are stopped after the playbook.

In [ ]:
zemi.arsenal.begin(router_mode_arsenal, 
    stop_before_begin=False,
    llama_router_mode=True,
)

router_mode_arsenal.llamas.primary.models.qwen
router_mode_arsenal.llamas.primary.models.smollm
# The second model is added without restarting the parent router.

zemi.arsenal.end(router_mode_arsenal, stop_after_end=True)

## Checking the Model Mode constraint

The source Arsenal has multiple models per server, so starting it without Router Mode predictably raises `ValueError` before the first server starts.

In [ ]:
try:
    zemi.arsenal.begin(router_mode_arsenal, 
        stop_before_begin=False,
        llama_router_mode=False,
    )
except ValueError as error:
    print(f"Expected configuration error: {error}")